#### 7. Land multiple CSV files with slightly different formats (e.g., an extra column, a different delimiter) and document how your read_files() options need to change for each, plus how you'd detect a mismatch before it silently breaks downstream reports.

In [0]:
SELECT * FROM read_files(
    '/Volumes/cyntexa_dev/sales/raw/emplyoees_comma.csv',
    format => 'csv',
    header => true,
    sep => ','
)

In [0]:
-- To detect the mismatch to avoid breaking silently, we can define the schema with rescuedDataColumn

select * from read_files(
    '/Volumes/cyntexa_dev/sales/raw/employees_extra_column.csv',
    format => 'csv',
    header => true,
    sep => ','
)

In [0]:
select * from read_files(
    '/Volumes/cyntexa_dev/sales/raw/employees_extra_column.csv',
    format => 'csv',
    header => true,
    sep => ',',
    schema => 'employee_id string,first_name string,last_name string,department string',
    rescuedDataColumn => '_rescued_data'
) 
where _rescued_data is not null

In [0]:
-- now we'll also save our data to not add false values if the format is changed 
-- This is a false querry to understand the results
select * from read_files(
    '/Volumes/cyntexa_dev/sales/raw/employees_pipe.csv',
    format => 'csv',
    header => true,
    sep => ','
) 

In [0]:
-- Correct one
select * from read_files(
    '/Volumes/cyntexa_dev/sales/raw/employees_pipe.csv',
    format => 'csv',
    header => true,
    sep => '|'
) 

#### 8. Write a decision memo: when should Cyntexa choose Iceberg (or Delta UniForm) over native Delta for a given table, considering downstream tools like Snowflake or Trino?

Decision Memo: Iceberg (or Delta UniForm) vs Native Delta for Cyntexa

EXECUTIVE SUMMARY:
Choose Iceberg (or Delta Uniform) when downstream tools (Snowflake, Trino, Presto, Athena) need direct table access.
Choose native Delta when the ecosystem is purely Databricks or uses Spark-compatible engines.

DECISION CRITERIA:

1. DOWNSTREAM TOOL REQUIREMENTS
   - Use Iceberg/UniForm if:
     * Snowflake, Trino, Presto, Athena, or other non-Spark engines must query the table directly
     * Business intelligence tools connect through these engines
     * Data sharing across heterogeneous platforms is required
     * Partners/customers use non-Databricks query engines

2. PERFORMANCE CONSIDERATIONS
   - Native Delta advantages:
     * Faster writes (no dual-format overhead)
     * Better Z-ordering and liquid clustering optimization
     * More efficient MERGE operations
     * Lower storage overhead (single format)
   - UniForm overhead:
     * Slight write latency increase (metadata conversion)
     * Additional storage for Iceberg metadata layer

3. FEATURE COMPATIBILITY
   - Native Delta exclusive features:
     * Change Data Feed (CDF) with full history
     * Deletion vectors for faster deletes
     * Advanced liquid clustering
     * Full Databricks optimization support
   - UniForm limitations:
     * Some Delta features may not translate to Iceberg readers
     * Schema evolution may require validation across formats

4. OPERATIONAL COMPLEXITY
   - Native Delta: Simpler operational model, single format maintenance
   - UniForm: Additional metadata layer management, format compatibility testing

5. COST ANALYSIS
   - Storage: UniForm incurs metadata duplication cost
   - Compute: Slight overhead on writes for format conversion
   - Data transfer: Evaluate if direct access reduces ETL/data movement costs

RECOMMENDATION FRAMEWORK:

USE ICEBERG/UNIFORM when:
  - Multi-engine access is a hard requirement
  - Downstream tools cannot use Databricks APIs/Spark
  - Data sharing/federation across platforms is core to architecture
  - Storage overhead acceptable relative to reduced ETL complexity

USE NATIVE DELTA when:
  - Databricks-centric or Spark-based ecosystem
  - Performance and cost optimization are priorities
  - Advanced Delta features (CDF, deletion vectors) are needed
  - Downstream systems can consume via Delta Sharing, Databricks SQL, or Spark connectors


#### 9. Use DESCRIBE HISTORY together with the metadata columns to trace a specific bad row back to the exact ingestion run and source file that introduced it.

In [0]:

select *, _metadata.file_modification_time 
from cyntexa_dev.sales.drivers_json
where driverId = 82;

In [0]:
describe history cyntexa_dev.sales.drivers_json